7  Movie Rating and Recommendation System

In [1]:
import pandas as pd

# Define a list of movies with genres
movies_data = {
    'Movie ID': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Title': [
        'The Action Hero',
        'Romantic Comedy',
        'Sci-Fi Epic',
        'Horror Fest',
        'Animated Adventure',
        'Drama King',
        'Space Odyssey',
        'Family Fun',
        'Thriller Night',
        'Historical Saga'
    ],
    'Genre': [
        'Action',
        'Comedy',
        'Sci-Fi',
        'Horror',
        'Animation',
        'Drama',
        'Sci-Fi',
        'Animation',
        'Thriller',
        'Drama'
    ]
}

movies_df = pd.DataFrame(movies_data)
display(movies_df.head())


,Movie ID,Title,Genre
0,1,The Action Hero,Action
1,2,Romantic Comedy,Comedy
2,3,Sci-Fi Epic,Sci-Fi
3,4,Horror Fest,Horror
4,5,Animated Adventure,Animation


## User Ratings
Now, let's simulate user ratings for some of these movies. Users will rate movies on a scale of 1 to 5.

In [ ]:
def get_user_ratings(movies_df):
    user_ratings = {}
    print("\n--- Rate some movies (1-5, or 's' to skip) ---")
    for index, row in movies_df.iterrows():
        while True:
            rating_input = input(f"Rate '{row['Title']}' ({row['Genre']}) [1-5, s=skip]: ").strip().lower()
            if rating_input == 's':
                break
            try:
                rating = int(rating_input)
                if 1 <= rating <= 5:
                    user_ratings[row['Movie ID']] = rating
                    break
                else:
                    print("Please enter a rating between 1 and 5.")
            except ValueError:
                print("Invalid input. Please enter a number or 's' to skip.")
    return user_ratings

# Get ratings from the user
current_user_ratings = get_user_ratings(movies_df)

print("\n--- Your Ratings ---")
if current_user_ratings:
    for movie_id, rating in current_user_ratings.items():
        movie_title = movies_df[movies_df['Movie ID'] == movie_id]['Title'].values[0]
        print(f"{movie_title}: {rating}/5")
else:
    print("No movies rated.")



--- Rate some movies (1-5, or 's' to skip) ---


MOVIE RECOMMENDATION


In [ ]:
def get_recommendations(user_ratings, movies_df, num_recommendations=3):
    if not user_ratings:
        print("Please rate some movies to get recommendations.")
        return pd.DataFrame()

    # Convert user ratings to a DataFrame for easier manipulation
    user_ratings_df = pd.DataFrame(list(user_ratings.items()), columns=['Movie ID', 'Rating'])

    # Identify highest-rated movies by the user
    highest_rated_movie_ids = user_ratings_df[user_ratings_df['Rating'] >= 4]['Movie ID'].tolist()

    if not highest_rated_movie_ids:
        print("Rate more movies highly (4 or 5 stars) to get genre-based recommendations.")
        # Fallback: recommend random unrated movies if no high ratings
        unrated_movie_ids = movies_df[~movies_df['Movie ID'].isin(user_ratings.keys())]['Movie ID'].tolist()
        if unrated_movie_ids:
            return movies_df[movies_df['Movie ID'].isin(unrated_movie_ids)].sample(min(num_recommendations, len(unrated_movie_ids)))
        else:
            return pd.DataFrame()

    # Get genres of highest-rated movies
    highest_rated_genres = movies_df[movies_df['Movie ID'].isin(highest_rated_movie_ids)]['Genre'].unique()

    # Filter for movies that are not yet rated by the user
    unrated_movies_df = movies_df[~movies_df['Movie ID'].isin(user_ratings.keys())]

    # Recommend movies with matching genres from the unrated list
    recommendations = unrated_movies_df[unrated_movies_df['Genre'].isin(highest_rated_genres)]

    # If there aren't enough genre-matched recommendations, fill with other unrated movies
    if len(recommendations) < num_recommendations:
        other_unrated = unrated_movies_df[~unrated_movies_df['Movie ID'].isin(recommendations['Movie ID'])]
        recommendations = pd.concat([
            recommendations,
            other_unrated.sample(min(num_recommendations - len(recommendations), len(other_unrated)))
        ])

    return recommendations.head(num_recommendations)

# Get and display recommendations
print("\n--- Your Recommendations ---")
recommendations_df = get_recommendations(current_user_ratings, movies_df)

if not recommendations_df.empty:
    display(recommendations_df[['Title', 'Genre']])
else:
    print("Could not generate recommendations at this time.")
